# Data Cleaning & ETL

## Objective

This notebook performs data cleaning, transformation, and preparation of the raw datasets for analytical modeling.

The main objectives are:

- Standardize data types
- Handle missing values
- Remove inconsistencies
- Create analytical tables following the dimensional model defined previously
- Export processed datasets for analytics consumption

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

In [3]:
customers = pd.read_csv(RAW_DATA_PATH / "olist_customers_dataset.csv")
orders = pd.read_csv(RAW_DATA_PATH / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DATA_PATH / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW_DATA_PATH / "olist_products_dataset.csv")
payments = pd.read_csv(RAW_DATA_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DATA_PATH / "olist_order_reviews_dataset.csv")
sellers = pd.read_csv(RAW_DATA_PATH / "olist_sellers_dataset.csv")

In [4]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]


for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

Missing values were analyzed according to business context. Delivery-related fields were preserved because null values may represent valid operational states.

## Data Type Validation

After standardizing date columns, the dataset structure is validated to ensure that datetime fields were correctly converted and are ready for analytical transformations.

In [7]:
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## Missing Values Analysis

Missing values were reviewed after type conversion to understand their business meaning.

Delivery-related columns may contain null values because orders can be canceled, unavailable, or may not have reached specific operational stages.

In [6]:
missing_orders = (
    orders[date_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_orders

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_purchase_timestamp            0
order_estimated_delivery_date       0
dtype: int64

## Cleaning Rules

The following cleaning decisions were applied:

- Preserve operational null values related to order lifecycle.
- Convert date fields to datetime format.
- Remove duplicated records only when duplicate identification does not represent a valid business event.
- Standardize column names during dimensional modeling.

## Column Standardization

Column names were standardized before creating analytical tables.

Prefixes were removed from dimension attributes to avoid redundancy and improve readability in the final star schema model.

In [8]:
customers = customers.rename(
    columns={
        "customer_id": "customer_id",
        "customer_unique_id": "customer_unique_id",
        "customer_zip_code_prefix": "zip_code_prefix",
        "customer_city": "city",
        "customer_state": "state"
    }
)

In [9]:
products = products.rename(
    columns={
        "product_category_name": "category_name",
        "product_name_length": "name_length",
        "product_description_length": "description_length",
        "product_photos_qty": "photos_qty",
        "product_weight_g": "weight_g",
        "product_length_cm": "length_cm",
        "product_height_cm": "height_cm",
        "product_width_cm": "width_cm"
    }
)

In [10]:
sellers = sellers.rename(
    columns={
        "seller_zip_code_prefix": "zip_code_prefix",
        "seller_city": "city",
        "seller_state": "state"
    }
)

In [11]:
customers.columns

Index(['customer_id', 'customer_unique_id', 'zip_code_prefix', 'city',
       'state'],
      dtype='str')

In [12]:
products.columns

Index(['product_id', 'category_name', 'product_name_lenght',
       'product_description_lenght', 'photos_qty', 'weight_g', 'length_cm',
       'height_cm', 'width_cm'],
      dtype='str')

In [13]:
sellers.columns

Index(['seller_id', 'zip_code_prefix', 'city', 'state'], dtype='str')